# audio-kri-russian-2 — Kaggle T4 pipeline (second dataset)

Transcribe a Russian lecture (Whisper **large-v3**) + diarize (pyannote) on a
**T4 GPU**, keep only the lecturer's clips, build a **parquet** dataset, and
push it to the Hugging Face Hub as **`audio-kri-russian-2`**.

This source audio is already **22 kHz**, so clips keep their **native sample
rate** — no resampling. The result clips are also zipped into the Kaggle output
so you can download them and drop them in your Google Drive `data/` folder to check.

**Before running:**
1. Notebook settings → Accelerator → **GPU T4 x1**.
2. Add your HF token as a Kaggle Secret named **`HF_TOKEN`**
   (Add-ons → Secrets, must be a **Write** token). The token's account must have
   accepted the licenses for `pyannote/speaker-diarization-community-1` and
   `pyannote/segmentation-3.0`.
3. Set `GDRIVE_URL` below to your Google Drive share link for the 22 kHz audio
   (file must be shared as "Anyone with the link").


In [ ]:
# 1. Install dependencies
!pip install -q openai-whisper pyannote.audio soundfile pydub gdown \
    "datasets>=2.18" huggingface_hub pyarrow


In [ ]:
# 2. Config
GDRIVE_URL    = "https://drive.google.com/file/d/REPLACE_WITH_FILE_ID/view?usp=sharing"
AUDIO_PATH    = "data/krimba_audio.flac"   # where we save the downloaded file

MODEL         = "large-v3"
LANGUAGE      = "ru"
DEVICE        = "cuda"                      # T4 GPU on Kaggle

DATASET_NAME  = "audio-kri-russian-2"      # second dataset
HF_USERNAME   = None                       # None -> auto-detect from token
PRIVATE_REPO  = True

# Filtering thresholds
MIN_LOGPROB        = -0.5    # min Whisper confidence
MAX_NO_SPEECH_PROB = 0.3     # max no-speech probability
MIN_DURATION       = 3.0     # seconds
MAX_DURATION       = 15.0    # seconds

CLIPS_DIR     = "clips"
PARQUET_PATH  = f"{DATASET_NAME}.parquet"
ZIP_PATH      = f"/kaggle/working/{DATASET_NAME}_clips"   # .zip appended by make_archive


In [ ]:
# 3. Hugging Face token (from Kaggle Secrets) + login
import os
from huggingface_hub import login, whoami

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN not found. Add it under Add-ons -> Secrets (name: HF_TOKEN)."
login(token=HF_TOKEN)

username = HF_USERNAME or whoami()["name"]
REPO_ID = f"{username}/{DATASET_NAME}"
print("Will push to:", REPO_ID)


In [ ]:
# 4. Download audio from Google Drive
import os, gdown
os.makedirs(os.path.dirname(AUDIO_PATH), exist_ok=True)
gdown.download(url=GDRIVE_URL, output=AUDIO_PATH, fuzzy=True, quiet=False)

import soundfile as sf
info = sf.info(AUDIO_PATH)
print(f"{AUDIO_PATH}: {info.duration:.1f}s, {info.samplerate} Hz, {info.channels} ch, {info.format}")


In [ ]:
# 5. Pipeline functions (GPU-enabled)
import torch
import soundfile as sf
import whisper
from pyannote.audio import Pipeline


def transcribe(audio_path, model_size, language, device):
    print(f"Loading Whisper '{model_size}' on {device}...")
    model = whisper.load_model(model_size, device=device)
    print("Transcribing...")
    result = model.transcribe(audio_path, language=language, verbose=False)
    segs = []
    for s in result["segments"]:
        text = s["text"].strip()
        if text:
            segs.append({
                "start": s["start"], "end": s["end"], "text": text,
                "avg_logprob": s.get("avg_logprob", 0.0),
                "no_speech_prob": s.get("no_speech_prob", 0.0),
            })
    print(f"Whisper: {len(segs)} segments.")
    return segs


def diarize(audio_path, hf_token, device):
    print("Loading pyannote pipeline...")
    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-community-1", token=hf_token
    )
    if pipeline is None:
        raise RuntimeError("pyannote returned None - check token + accepted licenses.")
    pipeline.to(torch.device(device))

    print("Running diarization...")
    data, sr = sf.read(audio_path, dtype="float32", always_2d=True)
    waveform = torch.from_numpy(data.T)
    output = pipeline({"waveform": waveform, "sample_rate": sr})

    if hasattr(output, "exclusive_speaker_diarization"):
        return output.exclusive_speaker_diarization
    if hasattr(output, "speaker_diarization"):
        return output.speaker_diarization
    return output


def get_turns(diarization):
    turns = [
        {"start": t.start, "end": t.end, "speaker": spk}
        for t, _, spk in diarization.itertracks(yield_label=True)
    ]
    turns.sort(key=lambda x: x["start"])
    totals = {}
    for t in turns:
        totals[t["speaker"]] = totals.get(t["speaker"], 0.0) + (t["end"] - t["start"])
    print("Speakers (sec):", {k: round(v, 1) for k, v in sorted(totals.items())})
    lecturer = max(totals, key=totals.get)
    print("Lecturer (most speech):", lecturer)
    return turns, lecturer


def build_clips(turns, segs, lecturer, min_lp, max_ns, min_d, max_d):
    clips = []
    for t in turns:
        if t["speaker"] != lecturer:
            continue
        dur = t["end"] - t["start"]
        if dur < min_d or dur > max_d:
            continue
        overlapping = []
        for s in segs:
            ov = min(s["end"], t["end"]) - max(s["start"], t["start"])
            if ov <= 0:
                continue
            if s["avg_logprob"] < min_lp or s["no_speech_prob"] > max_ns:
                continue
            overlapping.append(s)
        if not overlapping:
            continue
        text = " ".join(s["text"] for s in overlapping).strip()
        if not text:
            continue
        clips.append({
            "start": t["start"], "end": t["end"], "speaker": t["speaker"],
            "text": text,
            "avg_logprob": min(s["avg_logprob"] for s in overlapping),
            "no_speech_prob": max(s["no_speech_prob"] for s in overlapping),
        })
    print(f"Built {len(clips)} lecturer clips.")
    return clips


In [ ]:
# 6. Run the pipeline
segments = transcribe(AUDIO_PATH, MODEL, LANGUAGE, DEVICE)
diarization = diarize(AUDIO_PATH, HF_TOKEN, DEVICE)
turns, lecturer = get_turns(diarization)
clips = build_clips(turns, segments, lecturer,
                    MIN_LOGPROB, MAX_NO_SPEECH_PROB, MIN_DURATION, MAX_DURATION)


In [ ]:
# 7. Cut audio into clip wav files
import os
from pydub import AudioSegment

os.makedirs(CLIPS_DIR, exist_ok=True)
audio = AudioSegment.from_file(AUDIO_PATH)

records = []
for i, c in enumerate(clips):
    clip_id = f"clip_{i+1:04d}"
    wav_path = os.path.join(CLIPS_DIR, f"{clip_id}.wav")
    audio[int(c["start"]*1000):int(c["end"]*1000)].export(wav_path, format="wav")
    records.append({
        "audio": wav_path,
        "text": c["text"],
        "speaker": c["speaker"],
        "start": round(c["start"], 3),
        "end": round(c["end"], 3),
        "duration": round(c["end"] - c["start"], 3),
    })

print(f"Wrote {len(records)} clips to {CLIPS_DIR}/")


In [ ]:
# 8. Preview before pushing — inspect rows and listen to a clip
import pandas as pd
from IPython.display import Audio, display

df = pd.DataFrame(records)
print(f"Total clips: {len(df)}")
print(f"Total duration: {df['duration'].sum()/60:.1f} min")
print(f"Duration per clip: min {df['duration'].min():.1f}s, "
      f"mean {df['duration'].mean():.1f}s, max {df['duration'].max():.1f}s")

# Show first 10 rows (text + timing)
display(df[["start", "end", "duration", "text"]].head(10))

# Play a sample clip to verify it's the lecturer and the text matches
SAMPLE_IDX = 0   # change to listen to a different clip
print(f"\nClip {SAMPLE_IDX}:  {records[SAMPLE_IDX]['text']}")
display(Audio(records[SAMPLE_IDX]["audio"]))


In [ ]:
# 9. Build HF dataset, save parquet, push to Hub
from datasets import Dataset, Audio

ds = Dataset.from_list(records)
ds = ds.cast_column("audio", Audio())   # embeds wav bytes; keeps native sample rate
print(ds)

# Local parquet (audio embedded as bytes)
ds.to_parquet(PARQUET_PATH)
print("Saved local parquet:", PARQUET_PATH)

# Push to the Hub (stored as parquet shards there too)
ds.push_to_hub(REPO_ID, private=PRIVATE_REPO)
print("Pushed to:", f"https://huggingface.co/datasets/{REPO_ID}")


In [ ]:
# 10. Zip clips into the Kaggle output for manual download to Google Drive
import shutil

zip_file = shutil.make_archive(ZIP_PATH, "zip", CLIPS_DIR)
size_mb = os.path.getsize(zip_file) / 1e6
print(f"Zipped {len(records)} clips -> {zip_file} ({size_mb:.1f} MB)")
print("Download it from the Kaggle 'Output' / 'Data' panel on the right,")
print("then upload it into your Google Drive 'data/' folder to check.")


## Notes
- Output dataset columns: `audio` (native **22 kHz** mono), `text`, `speaker`, `start`, `end`, `duration`.
- Source is already 22 kHz, so clips are **not resampled** (`Audio()` keeps native rate).
- `exclusive_speaker_diarization` already removes overlapping speech, so clips are clean single-speaker.
- Clips are also zipped to `/kaggle/working/audio-kri-russian-2_clips.zip` — download from the
  Output panel and drop into your Drive `data/` folder to verify.
- To make the dataset public later: set `PRIVATE_REPO = False`.
- Re-running `push_to_hub` overwrites the dataset on the Hub.
